In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("examples zero to hero") \
    .master("local[*]") \
    .getOrCreate()

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 11:44:16 WARN Utils: Your hostname, codespaces-b99eb4, resolves to a loopback address: 127.0.0.1; using 10.0.1.96 instead (on interface eth0)
25/11/23 11:44:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 11:44:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SECTION 1 — BASIC DATAFRAME OPERATIONS

Creating sample datframe to practice!

In [6]:
from pyspark.sql.functions import col

data = [
    ("IT",      2023, 120),
    ("IT",      2024, 150),
    ("HR",      2023, 80),
    ("HR",      2024, 90),
    ("Finance", 2023, 200),
    ("Finance", 2024, 220)
]

columns = ["dept", "year", "revenue"]

df = spark.createDataFrame(data, schema=columns)
df.show()


+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|    120|
|     IT|2024|    150|
|     HR|2023|     80|
|     HR|2024|     90|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



show only 2

In [7]:
df.show(2)

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
|  IT|2023|    120|
|  IT|2024|    150|
+----+----+-------+
only showing top 2 rows


check datatypes and how schema look like

In [6]:
df.printSchema()

root
 |-- dept: string (nullable = true)
 |-- year: long (nullable = true)
 |-- revenue: long (nullable = true)



Get number of rows

In [7]:
df.count()

6

Get columns

In [8]:
df.columns

['dept', 'year', 'revenue']

selecting columns

In [10]:
df.select(col("dept")).show()

+-------+
|   dept|
+-------+
|     IT|
|     IT|
|     HR|
|     HR|
|Finance|
|Finance|
+-------+



In [11]:
df.select("dept","revenue").show()

+-------+-------+
|   dept|revenue|
+-------+-------+
|     IT|    120|
|     IT|    150|
|     HR|     80|
|     HR|     90|
|Finance|    200|
|Finance|    220|
+-------+-------+



Renaming column

In [13]:
df.withColumnRenamed("dept","department").show()

+----------+----+-------+
|department|year|revenue|
+----------+----+-------+
|        IT|2023|    120|
|        IT|2024|    150|
|        HR|2023|     80|
|        HR|2024|     90|
|   Finance|2023|    200|
|   Finance|2024|    220|
+----------+----+-------+



Filter rows

In [18]:
df1 = df.filter(col("revenue")>250)
df1.show()

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



filtering with AND condition

In [30]:
df1=df.filter(
    (col("dept") == "IT") &
    (col("revenue") < 100)
).show()

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



filtering with OR condition

In [8]:
df.filter((col("year")==2023) | (col("revenue")>100)).show()


+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|    120|
|     IT|2024|    150|
|     HR|2023|     80|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



IN condition

In [31]:
df.filter(col("dept").isin("IT")).show()

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
|  IT|2023|    120|
|  IT|2024|    150|
+----+----+-------+



NOT condition (~)

In [32]:
df.filter(~col("dept").isin("IT")).show()

+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     HR|2023|     80|
|     HR|2024|     90|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



Distinct values

In [11]:
df.select("dept").distinct().show()

+-------+
|   dept|
+-------+
|     IT|
|     HR|
|Finance|
+-------+



Drop a column

In [12]:
df.drop("year").show()


+-------+-------+
|   dept|revenue|
+-------+-------+
|     IT|    120|
|     IT|    150|
|     HR|     80|
|     HR|     90|
|Finance|    200|
|Finance|    220|
+-------+-------+



Add a new column

In [15]:
df.withColumn("rev_k", col("revenue")/1000).show()


+-------+----+-------+-----+
|   dept|year|revenue|rev_k|
+-------+----+-------+-----+
|     IT|2023|    120| 0.12|
|     IT|2024|    150| 0.15|
|     HR|2023|     80| 0.08|
|     HR|2024|     90| 0.09|
|Finance|2023|    200|  0.2|
|Finance|2024|    220| 0.22|
+-------+----+-------+-----+



Cast column type

In [17]:
df.withColumn("revenue", col("revenue").cast("double")).show()


+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|  120.0|
|     IT|2024|  150.0|
|     HR|2023|   80.0|
|     HR|2024|   90.0|
|Finance|2023|  200.0|
|Finance|2024|  220.0|
+-------+----+-------+



Replace nulls

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

spark = SparkSession.builder.master("local[*]").appName("null-practice").getOrCreate()

data = [
    (1, "Somesh",   "IT",       90000,   "2025-01-05"),
    (2, None,       "IT",       120000,  "2025-01-15"),   # name NULL
    (3, "Rohan",    None,       70000,   "2025-02-10"),   # dept NULL
    (4, "Nikhil",   "Finance",  None,    "2025-02-25"),   # salary NULL
    (5, "Priya",    "IT",       95000,   None),           # date NULL
    (6, None,       None,       None,    None),           # everything NULL
    (7, "John",     "Finance",  105000,  "2025-03-20"),
    (8, "Arjun",    "IT",       98000,   "2025-03-25"),
    (9, None,       "IT",       65000,   "2025-03-30"),   # name NULL
    (10, "Sandeep", "HR",       85000,   None)            # date NULL
]

columns = ["emp_id", "name", "dept", "salary", "join_date"]

df_nulls = spark.createDataFrame(data, columns)
df_nulls.show(truncate=False)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 08:22:17 WARN Utils: Your hostname, codespaces-b99eb4, resolves to a loopback address: 127.0.0.1; using 10.0.3.238 instead (on interface eth0)
25/11/23 08:22:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 08:22:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+-------+-------+------+----------+
|emp_id|name   |dept   |salary|join_date |
+------+-------+-------+------+----------+
|1     |Somesh |IT     |90000 |2025-01-05|
|2     |NULL   |IT     |120000|2025-01-15|
|3     |Rohan  |NULL   |70000 |2025-02-10|
|4     |Nikhil |Finance|NULL  |2025-02-25|
|5     |Priya  |IT     |95000 |NULL      |
|6     |NULL   |NULL   |NULL  |NULL      |
|7     |John   |Finance|105000|2025-03-20|
|8     |Arjun  |IT     |98000 |2025-03-25|
|9     |NULL   |IT     |65000 |2025-03-30|
|10    |Sandeep|HR     |85000 |NULL      |
+------+-------+-------+------+----------+



In [22]:
df_nulls.fillna({"dept":"N/A"}).show()


+------+-------+-------+------+----------+
|emp_id|   name|   dept|salary| join_date|
+------+-------+-------+------+----------+
|     1| Somesh|     IT| 90000|2025-01-05|
|     2|   NULL|     IT|120000|2025-01-15|
|     3|  Rohan|    N/A| 70000|2025-02-10|
|     4| Nikhil|Finance|  NULL|2025-02-25|
|     5|  Priya|     IT| 95000|      NULL|
|     6|   NULL|    N/A|  NULL|      NULL|
|     7|   John|Finance|105000|2025-03-20|
|     8|  Arjun|     IT| 98000|2025-03-25|
|     9|   NULL|     IT| 65000|2025-03-30|
|    10|Sandeep|     HR| 85000|      NULL|
+------+-------+-------+------+----------+



In [23]:
df_nulls.dropna().show()


+------+------+-------+------+----------+
|emp_id|  name|   dept|salary| join_date|
+------+------+-------+------+----------+
|     1|Somesh|     IT| 90000|2025-01-05|
|     7|  John|Finance|105000|2025-03-20|
|     8| Arjun|     IT| 98000|2025-03-25|
+------+------+-------+------+----------+



Order by revenue desc

In [32]:
df.orderBy(col("salary").desc()).show()


+------+-------+-------+------+----------+
|emp_id|   name|   dept|salary| join_date|
+------+-------+-------+------+----------+
|     2|   NULL|     IT|120000|2025-01-15|
|     7|   John|Finance|105000|2025-03-20|
|     8|  Arjun|     IT| 98000|2025-03-25|
|     5|  Priya|     IT| 95000|      NULL|
|     1| Somesh|     IT| 90000|2025-01-05|
|    10|Sandeep|     HR| 85000|      NULL|
|     3|  Rohan|   NULL| 70000|2025-02-10|
|     9|   NULL|     IT| 65000|2025-03-30|
|     4| Nikhil|Finance|  NULL|2025-02-25|
|     6|   NULL|   NULL|  NULL|      NULL|
+------+-------+-------+------+----------+



SECTION 2 — GROUPBY & AGGREGATIONS

Count rows per dept

In [7]:
df.groupBy("dept").count().show()


+-------+-----+
|   dept|count|
+-------+-----+
|     IT|    2|
|     HR|    2|
|Finance|    2|
+-------+-----+



Sum revenue per dept

In [9]:
from pyspark.sql.functions import sum
df.groupBy("dept").agg(sum("revenue").alias("total_rev")).show()


+-------+---------+
|   dept|total_rev|
+-------+---------+
|     IT|      270|
|     HR|      170|
|Finance|      420|
+-------+---------+



In [11]:
from pyspark.sql.functions import *

Max revenue

In [12]:
df.agg(max("revenue")).show()


+------------+
|max(revenue)|
+------------+
|         220|
+------------+



Min revenue

In [13]:
df.agg(min("revenue")).show()


+------------+
|min(revenue)|
+------------+
|          80|
+------------+



Multi-column groupBy

In [14]:
df.groupBy("dept","year").agg(sum("revenue")).show()


+-------+----+------------+
|   dept|year|sum(revenue)|
+-------+----+------------+
|     IT|2023|         120|
|     IT|2024|         150|
|     HR|2023|          80|
|     HR|2024|          90|
|Finance|2023|         200|
|Finance|2024|         220|
+-------+----+------------+



Count distinct years

In [16]:
from pyspark.sql.functions import countDistinct

df.select(countDistinct("year")).show()


+--------------------+
|count(DISTINCT year)|
+--------------------+
|                   2|
+--------------------+



Group and sort

In [17]:
df.groupBy("dept").agg(sum("revenue").alias("t")).orderBy(col("t").desc()).show()


+-------+---+
|   dept|  t|
+-------+---+
|Finance|420|
|     IT|270|
|     HR|170|
+-------+---+



Add constant column

In [18]:
df.withColumn("country", lit("India")).show()


+-------+----+-------+-------+
|   dept|year|revenue|country|
+-------+----+-------+-------+
|     IT|2023|    120|  India|
|     IT|2024|    150|  India|
|     HR|2023|     80|  India|
|     HR|2024|     90|  India|
|Finance|2023|    200|  India|
|Finance|2024|    220|  India|
+-------+----+-------+-------+



Bucket revenue into categories (when, otherwise)

In [19]:
df.withColumn("level", when(col("revenue")>150,"High").otherwise("Low")).show()


+-------+----+-------+-----+
|   dept|year|revenue|level|
+-------+----+-------+-----+
|     IT|2023|    120|  Low|
|     IT|2024|    150|  Low|
|     HR|2023|     80|  Low|
|     HR|2024|     90|  Low|
|Finance|2023|    200| High|
|Finance|2024|    220| High|
+-------+----+-------+-----+



Check duplicate rows

In [ ]:
df.groupBy(df.columns).count().filter(col("count")>1).show()


Collect rows into a list

In [20]:
df.groupBy("dept").agg(collect_list("revenue")).show()


+-------+---------------------+
|   dept|collect_list(revenue)|
+-------+---------------------+
|     IT|           [120, 150]|
|     HR|             [80, 90]|
|Finance|           [200, 220]|
+-------+---------------------+



Pivot

In [21]:
df.groupBy("dept").pivot("year").agg(sum("revenue")).show()


+-------+----+----+
|   dept|2023|2024|
+-------+----+----+
|     HR|  80|  90|
|Finance| 200| 220|
|     IT| 120| 150|
+-------+----+----+



SECTION 3 — JOINS

In [24]:
data = [
    ("IT",      "Ramesh",  500000),
    ("HR",      "Sita",    200000),
    ("Finance", "Karan",   800000)
]
columns = ["dept", "manager", "budget"]
df_dept = spark.createDataFrame(data, columns)


In [25]:
df_dept.show()

+-------+-------+------+
|   dept|manager|budget|
+-------+-------+------+
|     IT| Ramesh|500000|
|     HR|   Sita|200000|
|Finance|  Karan|800000|
+-------+-------+------+



Inner join

In [28]:
df.show()

+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|    120|
|     IT|2024|    150|
|     HR|2023|     80|
|     HR|2024|     90|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



In [29]:
df_dept.show()

+-------+-------+------+
|   dept|manager|budget|
+-------+-------+------+
|     IT| Ramesh|500000|
|     HR|   Sita|200000|
|Finance|  Karan|800000|
+-------+-------+------+



In [26]:
df.join(df_dept,"dept","inner").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
|     HR|2023|     80|   Sita|200000|
|     HR|2024|     90|   Sita|200000|
|     IT|2023|    120| Ramesh|500000|
|     IT|2024|    150| Ramesh|500000|
+-------+----+-------+-------+------+



Left Join

In [27]:
df.join(df_dept,"dept","left").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2023|    120| Ramesh|500000|
|     HR|2023|     80|   Sita|200000|
|     IT|2024|    150| Ramesh|500000|
|     HR|2024|     90|   Sita|200000|
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
+-------+----+-------+-------+------+



Right join

In [30]:
df.join(df_dept,"dept","right").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2024|    150| Ramesh|500000|
|     IT|2023|    120| Ramesh|500000|
|     HR|2024|     90|   Sita|200000|
|     HR|2023|     80|   Sita|200000|
|Finance|2024|    220|  Karan|800000|
|Finance|2023|    200|  Karan|800000|
+-------+----+-------+-------+------+



Full join

In [31]:
df.join(df_dept,"dept","outer").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
|     HR|2023|     80|   Sita|200000|
|     HR|2024|     90|   Sita|200000|
|     IT|2023|    120| Ramesh|500000|
|     IT|2024|    150| Ramesh|500000|
+-------+----+-------+-------+------+



Anti join - left_anti returns only those rows from the left DataFrame that have NO MATCH in the right DataFrame.

In [32]:
df.join(df_dept,"dept","left_anti").show()


+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



Semi join - left_semi = keep rows from left where a match exists in right

In [ ]:
df.join(df_dept,"dept","left_semi").show()


Broadcast join (optimize)

In [33]:
from pyspark.sql.functions import broadcast
df.join(broadcast(df_dept),"dept").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2023|    120| Ramesh|500000|
|     IT|2024|    150| Ramesh|500000|
|     HR|2023|     80|   Sita|200000|
|     HR|2024|     90|   Sita|200000|
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
+-------+----+-------+-------+------+



Self join

In [34]:
df.alias("a").join(df.alias("b"), col("a.dept")==col("b.dept")).show()


+-------+----+-------+-------+----+-------+
|   dept|year|revenue|   dept|year|revenue|
+-------+----+-------+-------+----+-------+
|Finance|2023|    200|Finance|2023|    200|
|Finance|2023|    200|Finance|2024|    220|
|Finance|2024|    220|Finance|2023|    200|
|Finance|2024|    220|Finance|2024|    220|
|     HR|2023|     80|     HR|2023|     80|
|     HR|2023|     80|     HR|2024|     90|
|     HR|2024|     90|     HR|2023|     80|
|     HR|2024|     90|     HR|2024|     90|
|     IT|2023|    120|     IT|2023|    120|
|     IT|2023|    120|     IT|2024|    150|
|     IT|2024|    150|     IT|2023|    120|
|     IT|2024|    150|     IT|2024|    150|
+-------+----+-------+-------+----+-------+



Check rows missing in dept table

In [36]:
df.join(df_dept,"dept","left_anti").show()


+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



Add default budget for missing dept

In [37]:
df.join(df_dept,"dept","left").fillna({"budget":0}).show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2023|    120| Ramesh|500000|
|     HR|2023|     80|   Sita|200000|
|     IT|2024|    150| Ramesh|500000|
|     HR|2024|     90|   Sita|200000|
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
+-------+----+-------+-------+------+



Multiple conditions

In [38]:
df.join(df_dept, (df.dept==df_dept.dept) & (df.year==2023)).show()


+-------+----+-------+-------+-------+------+
|   dept|year|revenue|   dept|manager|budget|
+-------+----+-------+-------+-------+------+
|Finance|2023|    200|Finance|  Karan|800000|
|     HR|2023|     80|     HR|   Sita|200000|
|     IT|2023|    120|     IT| Ramesh|500000|
+-------+----+-------+-------+-------+------+



SECTION 4 — WINDOW FUNCTIONS

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.master("local[*]").appName("window-practice").getOrCreate()

data = [
    ("IT",       "Somesh",  2023, 120),
    ("IT",       "Ramesh",  2024, 150),
    ("IT",       "Kiran",   2024, 180),

    ("HR",       "Sita",    2023,  80),
    ("HR",       "Anita",   2024,  90),
    ("HR",       "Rekha",   2024, 110),

    ("Finance",  "Karan",   2023, 200),
    ("Finance",  "John",    2024, 220),
    ("Finance",  "Amit",    2024, 250)
]

columns = ["dept", "employee", "year", "revenue"]

df = spark.createDataFrame(data, columns)
df.show()


25/11/23 11:50:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+--------+----+-------+
|   dept|employee|year|revenue|
+-------+--------+----+-------+
|     IT|  Somesh|2023|    120|
|     IT|  Ramesh|2024|    150|
|     IT|   Kiran|2024|    180|
|     HR|    Sita|2023|     80|
|     HR|   Anita|2024|     90|
|     HR|   Rekha|2024|    110|
|Finance|   Karan|2023|    200|
|Finance|    John|2024|    220|
|Finance|    Amit|2024|    250|
+-------+--------+----+-------+



Dense rank by revenue

dense_rank()
➜ Gives same rank to ties
➜ NO skipping of next number

In [7]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
w = Window.orderBy(col("revenue").desc())
df.withColumn("rank", dense_rank().over(w)).show()


25/11/23 11:51:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+----+
|   dept|employee|year|revenue|rank|
+-------+--------+----+-------+----+
|Finance|    Amit|2024|    250|   1|
|Finance|    John|2024|    220|   2|
|Finance|   Karan|2023|    200|   3|
|     IT|   Kiran|2024|    180|   4|
|     IT|  Ramesh|2024|    150|   5|
|     IT|  Somesh|2023|    120|   6|
|     HR|   Rekha|2024|    110|   7|
|     HR|   Anita|2024|     90|   8|
|     HR|    Sita|2023|     80|   9|
+-------+--------+----+-------+----+



row_number()
➜ Gives unique number to each row.
➜ NO ties allowed (even if values are same).

In [8]:
df.withColumn("rn", row_number().over(w)).show()


25/11/23 11:51:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+---+
|   dept|employee|year|revenue| rn|
+-------+--------+----+-------+---+
|Finance|    Amit|2024|    250|  1|
|Finance|    John|2024|    220|  2|
|Finance|   Karan|2023|    200|  3|
|     IT|   Kiran|2024|    180|  4|
|     IT|  Ramesh|2024|    150|  5|
|     IT|  Somesh|2023|    120|  6|
|     HR|   Rekha|2024|    110|  7|
|     HR|   Anita|2024|     90|  8|
|     HR|    Sita|2023|     80|  9|
+-------+--------+----+-------+---+



rank()
➜ Gives same rank to ties
➜ BUT skips the next number after a tie

In [9]:
df.withColumn("r", rank().over(w)).show()


25/11/23 11:51:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+---+
|   dept|employee|year|revenue|  r|
+-------+--------+----+-------+---+
|Finance|    Amit|2024|    250|  1|
|Finance|    John|2024|    220|  2|
|Finance|   Karan|2023|    200|  3|
|     IT|   Kiran|2024|    180|  4|
|     IT|  Ramesh|2024|    150|  5|
|     IT|  Somesh|2023|    120|  6|
|     HR|   Rekha|2024|    110|  7|
|     HR|   Anita|2024|     90|  8|
|     HR|    Sita|2023|     80|  9|
+-------+--------+----+-------+---+



Top 2 revenues

In [11]:
df.withColumn("rank", dense_rank().over(w)).filter(col("rank")<=2).show()


25/11/23 11:56:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:56:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:56:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+----+
|   dept|employee|year|revenue|rank|
+-------+--------+----+-------+----+
|Finance|    Amit|2024|    250|   1|
|Finance|    John|2024|    220|   2|
+-------+--------+----+-------+----+



25/11/23 11:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Partition by dept

In [12]:
w2 = Window.partitionBy("dept").orderBy(col("revenue"))
df.withColumn("rn", row_number().over(w2)).show()


+-------+--------+----+-------+---+
|   dept|employee|year|revenue| rn|
+-------+--------+----+-------+---+
|Finance|   Karan|2023|    200|  1|
|Finance|    John|2024|    220|  2|
|Finance|    Amit|2024|    250|  3|
|     HR|    Sita|2023|     80|  1|
|     HR|   Anita|2024|     90|  2|
|     HR|   Rekha|2024|    110|  3|
|     IT|  Somesh|2023|    120|  1|
|     IT|  Ramesh|2024|    150|  2|
|     IT|   Kiran|2024|    180|  3|
+-------+--------+----+-------+---+



Top 1 per dept

In [14]:
df.withColumn("rn", row_number().over(w2)).filter(col("rn")==1).show()


+-------+--------+----+-------+---+
|   dept|employee|year|revenue| rn|
+-------+--------+----+-------+---+
|Finance|   Karan|2023|    200|  1|
|     HR|    Sita|2023|     80|  1|
|     IT|  Somesh|2023|    120|  1|
+-------+--------+----+-------+---+



Cumulative sum revenue

In [17]:
wc = Window.orderBy("year").rowsBetween(Window.unboundedPreceding,Window.currentRow)
df.withColumn("running_total", sum("revenue").over(wc)).show()


25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+-------------+
|   dept|employee|year|revenue|running_total|
+-------+--------+----+-------+-------------+
|     IT|  Somesh|2023|    120|          120|
|     HR|    Sita|2023|     80|          200|
|Finance|   Karan|2023|    200|          400|
|     IT|  Ramesh|2024|    150|          550|
|     IT|   Kiran|2024|    180|          730|
|     HR|   Anita|2024|     90|          820|
|     HR|   Rekha|2024|    110|          930|
|Finance|    John|2024|    220|         1150|
|Finance|    Amit|2024|    250|         1400|
+-------+--------+----+-------+-------------+



25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [16]:
wc = Window.orderBy("year")
df.withColumn("running_total", sum("revenue").over(wc)).show()


25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+-------------+
|   dept|employee|year|revenue|running_total|
+-------+--------+----+-------+-------------+
|     IT|  Somesh|2023|    120|          400|
|     HR|    Sita|2023|     80|          400|
|Finance|   Karan|2023|    200|          400|
|     IT|  Ramesh|2024|    150|         1400|
|     IT|   Kiran|2024|    180|         1400|
|     HR|   Anita|2024|     90|         1400|
|     HR|   Rekha|2024|    110|         1400|
|Finance|    John|2024|    220|         1400|
|Finance|    Amit|2024|    250|         1400|
+-------+--------+----+-------+-------------+



25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Lag revenue

In [20]:
df.withColumn("prev_rev", lag("revenue").over(w)).show()


25/11/23 12:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+--------+
|   dept|employee|year|revenue|prev_rev|
+-------+--------+----+-------+--------+
|Finance|    Amit|2024|    250|    NULL|
|Finance|    John|2024|    220|     250|
|Finance|   Karan|2023|    200|     220|
|     IT|   Kiran|2024|    180|     200|
|     IT|  Ramesh|2024|    150|     180|
|     IT|  Somesh|2023|    120|     150|
|     HR|   Rekha|2024|    110|     120|
|     HR|   Anita|2024|     90|     110|
|     HR|    Sita|2023|     80|      90|
+-------+--------+----+-------+--------+



25/11/23 12:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Lead revenue

In [21]:
df.withColumn("next_rev", lead("revenue").over(w)).show()


25/11/23 12:47:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:47:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:47:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+--------+
|   dept|employee|year|revenue|next_rev|
+-------+--------+----+-------+--------+
|Finance|    Amit|2024|    250|     220|
|Finance|    John|2024|    220|     200|
|Finance|   Karan|2023|    200|     180|
|     IT|   Kiran|2024|    180|     150|
|     IT|  Ramesh|2024|    150|     120|
|     IT|  Somesh|2023|    120|     110|
|     HR|   Rekha|2024|    110|      90|
|     HR|   Anita|2024|     90|      80|
|     HR|    Sita|2023|     80|    NULL|
+-------+--------+----+-------+--------+



25/11/23 12:47:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:47:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Calculate difference with previous revenue

In [23]:
df.withColumn("prev", lag("revenue").over(w))\
  .withColumn("diff", col("revenue")-col("prev")).show()


25/11/23 12:48:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:48:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:48:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+----+----+
|   dept|employee|year|revenue|prev|diff|
+-------+--------+----+-------+----+----+
|Finance|    Amit|2024|    250|NULL|NULL|
|Finance|    John|2024|    220| 250| -30|
|Finance|   Karan|2023|    200| 220| -20|
|     IT|   Kiran|2024|    180| 200| -20|
|     IT|  Ramesh|2024|    150| 180| -30|
|     IT|  Somesh|2023|    120| 150| -30|
|     HR|   Rekha|2024|    110| 120| -10|
|     HR|   Anita|2024|     90| 110| -20|
|     HR|    Sita|2023|     80|  90| -10|
+-------+--------+----+-------+----+----+



25/11/23 12:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Ntile split

In [24]:
df.withColumn("bucket", ntile(4).over(w)).show()


25/11/23 12:48:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:48:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:48:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+------+
|   dept|employee|year|revenue|bucket|
+-------+--------+----+-------+------+
|Finance|    Amit|2024|    250|     1|
|Finance|    John|2024|    220|     1|
|Finance|   Karan|2023|    200|     1|
|     IT|   Kiran|2024|    180|     2|
|     IT|  Ramesh|2024|    150|     2|
|     IT|  Somesh|2023|    120|     3|
|     HR|   Rekha|2024|    110|     3|
|     HR|   Anita|2024|     90|     4|
|     HR|    Sita|2023|     80|     4|
+-------+--------+----+-------+------+



25/11/23 12:48:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:48:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Highest revenue per year

In [25]:
w3 = Window.partitionBy("year").orderBy(col("revenue").desc())
df.withColumn("rk", row_number().over(w3)).filter(col("rk")==1).show()


+-------+--------+----+-------+---+
|   dept|employee|year|revenue| rk|
+-------+--------+----+-------+---+
|Finance|   Karan|2023|    200|  1|
|Finance|    Amit|2024|    250|  1|
+-------+--------+----+-------+---+



Average revenue per dept 

In [26]:
w4 = Window.partitionBy("dept")
df.withColumn("avg_dep", avg("revenue").over(w4)).show()


+-------+--------+----+-------+------------------+
|   dept|employee|year|revenue|           avg_dep|
+-------+--------+----+-------+------------------+
|Finance|   Karan|2023|    200|223.33333333333334|
|Finance|    John|2024|    220|223.33333333333334|
|Finance|    Amit|2024|    250|223.33333333333334|
|     HR|    Sita|2023|     80| 93.33333333333333|
|     HR|   Anita|2024|     90| 93.33333333333333|
|     HR|   Rekha|2024|    110| 93.33333333333333|
|     IT|  Somesh|2023|    120|             150.0|
|     IT|  Ramesh|2024|    150|             150.0|
|     IT|   Kiran|2024|    180|             150.0|
+-------+--------+----+-------+------------------+



Difference between highest and lowest per dept

In [27]:
df.groupBy("dept").agg((max("revenue")-min("revenue")).alias("gap")).show()


+-------+---+
|   dept|gap|
+-------+---+
|     IT| 60|
|     HR| 30|
|Finance| 50|
+-------+---+



SECTION 5 — DATE & TIME FUNCTIONS

Convert to_date

In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

spark = SparkSession.builder.master("local[*]").appName("date-complex-practice").getOrCreate()

data = [
    # dept, employee, year, revenue, join_date (ISO string)
    ("IT",      "Somesh", 2025, 120000, "2025-11-10"),
    ("IT",      "Ramesh", 2024, 150000, "2024-10-05"),
    ("IT",      "Kiran",  2023, 180000, "2023-12-20"),

    ("HR",      "Sita",   2025,  80000, "2025-10-30"),
    ("HR",      "Anita",  2024,  90000, "2024-11-15"),
    ("HR",      "Rekha",  2023, 110000, "2023-09-01"),

    ("Finance", "Karan",  2025, 200000, "2025-11-01"),
    ("Finance", "John",   2024, 220000, "2024-07-20"),
    ("Finance", "Amit",   2022, 250000, "2022-05-10"),

    # some rows with null / edge cases
    ("Admin",   None,     2024,   None,  None),
    ("IT",      "Guest",  2021,  50000, "2021-12-31")
]

columns = ["dept", "employee", "year", "revenue", "join_date"]

df = spark.createDataFrame(data, schema=columns)

# show the DF
df.show(truncate=False)


25/11/23 12:53:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+--------+----+-------+----------+
|dept   |employee|year|revenue|join_date |
+-------+--------+----+-------+----------+
|IT     |Somesh  |2025|120000 |2025-11-10|
|IT     |Ramesh  |2024|150000 |2024-10-05|
|IT     |Kiran   |2023|180000 |2023-12-20|
|HR     |Sita    |2025|80000  |2025-10-30|
|HR     |Anita   |2024|90000  |2024-11-15|
|HR     |Rekha   |2023|110000 |2023-09-01|
|Finance|Karan   |2025|200000 |2025-11-01|
|Finance|John    |2024|220000 |2024-07-20|
|Finance|Amit    |2022|250000 |2022-05-10|
|Admin  |NULL    |2024|NULL   |NULL      |
|IT     |Guest   |2021|50000  |2021-12-31|
+-------+--------+----+-------+----------+



In [33]:
df.printSchema()

root
 |-- dept: string (nullable = true)
 |-- employee: string (nullable = true)
 |-- year: long (nullable = true)
 |-- revenue: long (nullable = true)
 |-- join_date: string (nullable = true)



Convert to_date:
By default, to_date() expects the string in format:
yyyy-MM-dd

In [46]:
from pyspark.sql.functions import to_date, col

df2 = df.withColumn("join_date", to_date(col("join_date"), "yyyy-MM-dd"))
df2.show()


+-------+--------+----+-------+----------+
|   dept|employee|year|revenue| join_date|
+-------+--------+----+-------+----------+
|     IT|  Somesh|2025| 120000|2025-11-10|
|     IT|  Ramesh|2024| 150000|2024-10-05|
|     IT|   Kiran|2023| 180000|2023-12-20|
|     HR|    Sita|2025|  80000|2025-10-30|
|     HR|   Anita|2024|  90000|2024-11-15|
|     HR|   Rekha|2023| 110000|2023-09-01|
|Finance|   Karan|2025| 200000|2025-11-01|
|Finance|    John|2024| 220000|2024-07-20|
|Finance|    Amit|2022| 250000|2022-05-10|
|  Admin|    NULL|2024|   NULL|      NULL|
|     IT|   Guest|2021|  50000|2021-12-31|
+-------+--------+----+-------+----------+



withColumn replaces the column if it already exists.

In [45]:
df2.printSchema()

root
 |-- dept: string (nullable = true)
 |-- employee: string (nullable = true)
 |-- year: long (nullable = true)
 |-- revenue: long (nullable = true)
 |-- join_date: date (nullable = true)



In [49]:
from pyspark.sql.functions import to_date, col

df3 = df.withColumn("date", to_date(col("join_date"), "yyyy-MM-dd")).show(2)


+----+--------+----+-------+----------+----------+
|dept|employee|year|revenue| join_date|      date|
+----+--------+----+-------+----------+----------+
|  IT|  Somesh|2025| 120000|2025-11-10|2025-11-10|
|  IT|  Ramesh|2024| 150000|2024-10-05|2024-10-05|
+----+--------+----+-------+----------+----------+
only showing top 2 rows


In [43]:
df3.printSchema()

root
 |-- dept: string (nullable = true)
 |-- employee: string (nullable = true)
 |-- year: long (nullable = true)
 |-- revenue: long (nullable = true)
 |-- join_date: string (nullable = true)
 |-- date: date (nullable = true)



current_date

In [51]:
df.withColumn("today", current_date()).show(2)


+----+--------+----+-------+----------+----------+
|dept|employee|year|revenue| join_date|     today|
+----+--------+----+-------+----------+----------+
|  IT|  Somesh|2025| 120000|2025-11-10|2025-11-23|
|  IT|  Ramesh|2024| 150000|2024-10-05|2025-11-23|
+----+--------+----+-------+----------+----------+
only showing top 2 rows


datediff

In [52]:
df.withColumn("age_days", datediff(current_date(), to_date(lit("2023-01-01")))).show()


+-------+--------+----+-------+----------+--------+
|   dept|employee|year|revenue| join_date|age_days|
+-------+--------+----+-------+----------+--------+
|     IT|  Somesh|2025| 120000|2025-11-10|    1057|
|     IT|  Ramesh|2024| 150000|2024-10-05|    1057|
|     IT|   Kiran|2023| 180000|2023-12-20|    1057|
|     HR|    Sita|2025|  80000|2025-10-30|    1057|
|     HR|   Anita|2024|  90000|2024-11-15|    1057|
|     HR|   Rekha|2023| 110000|2023-09-01|    1057|
|Finance|   Karan|2025| 200000|2025-11-01|    1057|
|Finance|    John|2024| 220000|2024-07-20|    1057|
|Finance|    Amit|2022| 250000|2022-05-10|    1057|
|  Admin|    NULL|2024|   NULL|      NULL|    1057|
|     IT|   Guest|2021|  50000|2021-12-31|    1057|
+-------+--------+----+-------+----------+--------+



date_add

In [53]:
df.withColumn("future", date_add(current_date(), 10)).show()


+-------+--------+----+-------+----------+----------+
|   dept|employee|year|revenue| join_date|    future|
+-------+--------+----+-------+----------+----------+
|     IT|  Somesh|2025| 120000|2025-11-10|2025-12-03|
|     IT|  Ramesh|2024| 150000|2024-10-05|2025-12-03|
|     IT|   Kiran|2023| 180000|2023-12-20|2025-12-03|
|     HR|    Sita|2025|  80000|2025-10-30|2025-12-03|
|     HR|   Anita|2024|  90000|2024-11-15|2025-12-03|
|     HR|   Rekha|2023| 110000|2023-09-01|2025-12-03|
|Finance|   Karan|2025| 200000|2025-11-01|2025-12-03|
|Finance|    John|2024| 220000|2024-07-20|2025-12-03|
|Finance|    Amit|2022| 250000|2022-05-10|2025-12-03|
|  Admin|    NULL|2024|   NULL|      NULL|2025-12-03|
|     IT|   Guest|2021|  50000|2021-12-31|2025-12-03|
+-------+--------+----+-------+----------+----------+



datediff gives the count of days between two dates.
date_sub subtracts days from a date and returns a new date.

date_sub

In [54]:
df.withColumn("past", date_sub(current_date(), 30)).show()


+-------+--------+----+-------+----------+----------+
|   dept|employee|year|revenue| join_date|      past|
+-------+--------+----+-------+----------+----------+
|     IT|  Somesh|2025| 120000|2025-11-10|2025-10-24|
|     IT|  Ramesh|2024| 150000|2024-10-05|2025-10-24|
|     IT|   Kiran|2023| 180000|2023-12-20|2025-10-24|
|     HR|    Sita|2025|  80000|2025-10-30|2025-10-24|
|     HR|   Anita|2024|  90000|2024-11-15|2025-10-24|
|     HR|   Rekha|2023| 110000|2023-09-01|2025-10-24|
|Finance|   Karan|2025| 200000|2025-11-01|2025-10-24|
|Finance|    John|2024| 220000|2024-07-20|2025-10-24|
|Finance|    Amit|2022| 250000|2022-05-10|2025-10-24|
|  Admin|    NULL|2024|   NULL|      NULL|2025-10-24|
|     IT|   Guest|2021|  50000|2021-12-31|2025-10-24|
+-------+--------+----+-------+----------+----------+



month extract, year extract

In [56]:
df.withColumn("month", month(current_date())).show(2)
df.withColumn("yr", year(current_date())).show(2)


+----+--------+----+-------+----------+-----+
|dept|employee|year|revenue| join_date|month|
+----+--------+----+-------+----------+-----+
|  IT|  Somesh|2025| 120000|2025-11-10|   11|
|  IT|  Ramesh|2024| 150000|2024-10-05|   11|
+----+--------+----+-------+----------+-----+
only showing top 2 rows
+----+--------+----+-------+----------+----+
|dept|employee|year|revenue| join_date|  yr|
+----+--------+----+-------+----------+----+
|  IT|  Somesh|2025| 120000|2025-11-10|2025|
|  IT|  Ramesh|2024| 150000|2024-10-05|2025|
+----+--------+----+-------+----------+----+
only showing top 2 rows


date_format

In [58]:
df.withColumn("formatted", date_format(current_date(),"yyyy-MM-dd")).show(2)


+----+--------+----+-------+----------+----------+
|dept|employee|year|revenue| join_date| formatted|
+----+--------+----+-------+----------+----------+
|  IT|  Somesh|2025| 120000|2025-11-10|2025-11-23|
|  IT|  Ramesh|2024| 150000|2024-10-05|2025-11-23|
+----+--------+----+-------+----------+----------+
only showing top 2 rows


employees joined last 30 days

In [59]:
df.filter(col("join_date") >= date_sub(current_date(),30)).show()


+-------+--------+----+-------+----------+
|   dept|employee|year|revenue| join_date|
+-------+--------+----+-------+----------+
|     IT|  Somesh|2025| 120000|2025-11-10|
|     HR|    Sita|2025|  80000|2025-10-30|
|Finance|   Karan|2025| 200000|2025-11-01|
+-------+--------+----+-------+----------+



SECTION 6 — COMPLEX TYPES

In [61]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

spark = SparkSession.builder.master("local[*]").appName("complex-types").getOrCreate()

data = [
    ("IT",      "Somesh", 2025, 120000),
    ("HR",      "Anita",  2024,  90000),
    ("Finance", "Karan",  2023, 200000),
    ("Admin",   "John",   2022,  50000),
    ("IT",      "Kiran",  2021, 150000),
]

columns = ["dept", "employee", "year", "revenue"]

df = spark.createDataFrame(data, columns)
df.show()


25/11/23 13:09:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+--------+----+-------+
|   dept|employee|year|revenue|
+-------+--------+----+-------+
|     IT|  Somesh|2025| 120000|
|     HR|   Anita|2024|  90000|
|Finance|   Karan|2023| 200000|
|  Admin|    John|2022|  50000|
|     IT|   Kiran|2021| 150000|
+-------+--------+----+-------+



Create array

RULE: All elements inside array() must have the SAME data type

In [63]:
from pyspark.sql.functions import array, col

df.withColumn("arr", array(col("dept"), col("year").cast("string"))).show()


+-------+--------+----+-------+---------------+
|   dept|employee|year|revenue|            arr|
+-------+--------+----+-------+---------------+
|     IT|  Somesh|2025| 120000|     [IT, 2025]|
|     HR|   Anita|2024|  90000|     [HR, 2024]|
|Finance|   Karan|2023| 200000|[Finance, 2023]|
|  Admin|    John|2022|  50000|  [Admin, 2022]|
|     IT|   Kiran|2021| 150000|     [IT, 2021]|
+-------+--------+----+-------+---------------+



Explode array

In [64]:
df2 = df.withColumn("arr", array("year","revenue"))
df2.select("dept", explode("arr")).show()


+-------+------+
|   dept|   col|
+-------+------+
|     IT|  2025|
|     IT|120000|
|     HR|  2024|
|     HR| 90000|
|Finance|  2023|
|Finance|200000|
|  Admin|  2022|
|  Admin| 50000|
|     IT|  2021|
|     IT|150000|
+-------+------+



Split string

In [76]:
df2 = df.withColumn("dept_split", split(col("dept"),""))
df2.show(truncate=False)


+-------+--------+----+-------+---------------------+
|dept   |employee|year|revenue|dept_split           |
+-------+--------+----+-------+---------------------+
|IT     |Somesh  |2025|120000 |[I, T]               |
|HR     |Anita   |2024|90000  |[H, R]               |
|Finance|Karan   |2023|200000 |[F, i, n, a, n, c, e]|
|Admin  |John    |2022|50000  |[A, d, m, i, n]      |
|IT     |Kiran   |2021|150000 |[I, T]               |
+-------+--------+----+-------+---------------------+



Get array element

In [77]:
df2.select(col("dept_split")[0]).show()


+-------------+
|dept_split[0]|
+-------------+
|            I|
|            H|
|            F|
|            A|
|            I|
+-------------+



Combine multiple values

In [78]:
df.withColumn("combo", concat(col("dept"), lit("-"), col("year"))).show()


+-------+--------+----+-------+------------+
|   dept|employee|year|revenue|       combo|
+-------+--------+----+-------+------------+
|     IT|  Somesh|2025| 120000|     IT-2025|
|     HR|   Anita|2024|  90000|     HR-2024|
|Finance|   Karan|2023| 200000|Finance-2023|
|  Admin|    John|2022|  50000|  Admin-2022|
|     IT|   Kiran|2021| 150000|     IT-2021|
+-------+--------+----+-------+------------+



Reverse array


In [79]:
df2.withColumn("rev_arr", reverse(col("dept_split"))).show()


+-------+--------+----+-------+--------------------+--------------------+
|   dept|employee|year|revenue|          dept_split|             rev_arr|
+-------+--------+----+-------+--------------------+--------------------+
|     IT|  Somesh|2025| 120000|              [I, T]|              [T, I]|
|     HR|   Anita|2024|  90000|              [H, R]|              [R, H]|
|Finance|   Karan|2023| 200000|[F, i, n, a, n, c...|[e, c, n, a, n, i...|
|  Admin|    John|2022|  50000|     [A, d, m, i, n]|     [n, i, m, d, A]|
|     IT|   Kiran|2021| 150000|              [I, T]|              [T, I]|
+-------+--------+----+-------+--------------------+--------------------+



Cache DataFrame: cache() stores a DataFrame in memory so Spark doesn’t have to recalculate it every time you use it.

In [80]:
df.cache()
df.count()


5

Use repartition() when increasing partitions, to increase parallelism, You want balanced load across workers

In [81]:
df.repartition(3)


DataFrame[dept: string, employee: string, year: bigint, revenue: bigint]

Use coalesce() when decreasing partitions faster
coalesce() merges partitions together
It does NOT shuffle data
So decreasing partitions becomes FAST

In [82]:
df.coalesce(1)


DataFrame[dept: string, employee: string, year: bigint, revenue: bigint]

Explain Plan = Spark’s blueprint / roadmap of how the job will run.

It shows:What operations Spark will perform, Which steps need shuffles, Which filters / projections Spark applies, How joins are planned, How data moves between partitions


In [83]:
df.explain(True)


== Parsed Logical Plan ==
LogicalRDD [dept#805, employee#806, year#807L, revenue#808L], false

== Analyzed Logical Plan ==
dept: string, employee: string, year: bigint, revenue: bigint
LogicalRDD [dept#805, employee#806, year#807L, revenue#808L], false

== Optimized Logical Plan ==
InMemoryRelation [dept#805, employee#806, year#807L, revenue#808L], StorageLevel(disk, memory, deserialized, 1 replicas)
   +- *(1) Scan ExistingRDD[dept#805,employee#806,year#807L,revenue#808L]

== Physical Plan ==
InMemoryTableScan [dept#805, employee#806, year#807L, revenue#808L]
   +- InMemoryRelation [dept#805, employee#806, year#807L, revenue#808L], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Scan ExistingRDD[dept#805,employee#806,year#807L,revenue#808L]



In [85]:
df.printSchema()

root
 |-- dept: string (nullable = true)
 |-- employee: string (nullable = true)
 |-- year: long (nullable = true)
 |-- revenue: long (nullable = true)



In [86]:
df.filter(col("revenue") > 50000).select("employee").explain()


== Physical Plan ==
*(1) Project [employee#806]
+- *(1) Filter (isnotnull(revenue#808L) AND (revenue#808L > 50000))
   +- InMemoryTableScan [employee#806, revenue#808L], [isnotnull(revenue#808L), (revenue#808L > 50000)]
         +- InMemoryRelation [dept#805, employee#806, year#807L, revenue#808L], StorageLevel(disk, memory, deserialized, 1 replicas)
               +- *(1) Scan ExistingRDD[dept#805,employee#806,year#807L,revenue#808L]




Write Parquet

df.write.mode("overwrite").parquet("outputpath")


Read Parquet

spark.read.parquet("out/pq").show()


Write CSV

df.write.mode("overwrite").option("header",True).csv("out/csv")


Read CSV

spark.read.option("header",True).csv("out/csv").show()


Rename multiple columns at once

In [87]:
df2 = df.withColumnRenamed("dept","department") \
        .withColumnRenamed("revenue","rev")


Select all columns except specific ones

In [88]:
cols = [c for c in df.columns if c not in ("year")]
df.select(cols).show()


+-------+--------+-------+
|   dept|employee|revenue|
+-------+--------+-------+
|     IT|  Somesh| 120000|
|     HR|   Anita|  90000|
|Finance|   Karan| 200000|
|  Admin|    John|  50000|
|     IT|   Kiran| 150000|
+-------+--------+-------+



Replace negative values with 0

In [ ]:
df.withColumn("revenue", when(col("revenue") < 0, 0).otherwise(col("revenue"))).show()


Replace string values

Replacing IT with InformationTech in dept column

In [90]:
df.replace({"IT": "InformationTech"}, subset=["dept"]).show()


+---------------+--------+----+-------+
|           dept|employee|year|revenue|
+---------------+--------+----+-------+
|InformationTech|  Somesh|2025| 120000|
|             HR|   Anita|2024|  90000|
|        Finance|   Karan|2023| 200000|
|          Admin|    John|2022|  50000|
|InformationTech|   Kiran|2021| 150000|
+---------------+--------+----+-------+



Filter using SQL expression

In [92]:
df.filter("revenue > 120").show(2)


+----+--------+----+-------+
|dept|employee|year|revenue|
+----+--------+----+-------+
|  IT|  Somesh|2025| 120000|
|  HR|   Anita|2024|  90000|
+----+--------+----+-------+
only showing top 2 rows


Multi-column orderBy

First sort by dept
Then within that dept, sort by revenue descending
ex. lets say 2 fields having IT then goes with revenue 

In [93]:
df.orderBy(col("dept"), col("revenue").desc()).show()


+-------+--------+----+-------+
|   dept|employee|year|revenue|
+-------+--------+----+-------+
|  Admin|    John|2022|  50000|
|Finance|   Karan|2023| 200000|
|     HR|   Anita|2024|  90000|
|     IT|   Kiran|2021| 150000|
|     IT|  Somesh|2025| 120000|
+-------+--------+----+-------+



Combine multiple DataFrames vertically

union() and unionAll() behave the SAME in PySpark.
unionByName() is different and more powerful.

Union()

Combines DataFrames vertically (rows).

Column order must be SAME.

Duplicates are NOT removed (unlike SQL UNION).

In [97]:
df3 = df1.union(df2)


unionAll()

Same as union() (in PySpark).

Can use union() instead.

In PySpark, union() and unionAll() work the same — both require the DataFrames to have the same columns in the same order. On the other hand, unionByName() is more flexible because it matches columns by their names instead of their order. That makes unionByName() the best choice when your DataFrames have the same column names but in different orders or when some columns are missing.

unionByName()

This is the powerful one.

Matches columns by NAME, not position.

Column order can be different.

In [94]:
df1 = spark.createDataFrame([
    (1, "A"),
    (2, "B")
], ["id", "name"])


In [95]:
df2 = spark.createDataFrame([
    (3, "C"),
    (4, "D")
], ["id", "name"])


In [96]:
df3 = df1.unionByName(df2)
df3.show()


+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
|  3|   C|
|  4|   D|
+---+----+



Remove duplicate rows

In [98]:
df.dropDuplicates().show()


+-------+--------+----+-------+
|   dept|employee|year|revenue|
+-------+--------+----+-------+
|     IT|  Somesh|2025| 120000|
|     HR|   Anita|2024|  90000|
|Finance|   Karan|2023| 200000|
|  Admin|    John|2022|  50000|
|     IT|   Kiran|2021| 150000|
+-------+--------+----+-------+



Remove duplicates based on selected columns

In [99]:
df.dropDuplicates(["dept"]).show()


+-------+--------+----+-------+
|   dept|employee|year|revenue|
+-------+--------+----+-------+
|  Admin|    John|2022|  50000|
|Finance|   Karan|2023| 200000|
|     HR|   Anita|2024|  90000|
|     IT|  Somesh|2025| 120000|
+-------+--------+----+-------+



Add a row_id column (monotonically increasing)

It generates a unique, increasing 64-bit ID for each row.

IDs are unique

IDs increase, but NOT sequential

In [102]:
df.withColumn("row_id", monotonically_increasing_id()).show()


+-------+--------+----+-------+-----------+
|   dept|employee|year|revenue|     row_id|
+-------+--------+----+-------+-----------+
|     IT|  Somesh|2025| 120000|          0|
|     HR|   Anita|2024|  90000| 8589934592|
|Finance|   Karan|2023| 200000|17179869184|
|  Admin|    John|2022|  50000|25769803776|
|     IT|   Kiran|2021| 150000|25769803777|
+-------+--------+----+-------+-----------+



Conditional multiple-when logic

In [104]:
df.withColumn("category",
    when(col("revenue")<100000,"LOW")
   .when(col("revenue")<150000,"MED")
   .otherwise("HIGH")
).show()


+-------+--------+----+-------+--------+
|   dept|employee|year|revenue|category|
+-------+--------+----+-------+--------+
|     IT|  Somesh|2025| 120000|     MED|
|     HR|   Anita|2024|  90000|     LOW|
|Finance|   Karan|2023| 200000|    HIGH|
|  Admin|    John|2022|  50000|     LOW|
|     IT|   Kiran|2021| 150000|    HIGH|
+-------+--------+----+-------+--------+



Add random column

In [107]:
df.withColumn("rand", rand()).show(2)


+----+--------+----+-------+------------------+
|dept|employee|year|revenue|              rand|
+----+--------+----+-------+------------------+
|  IT|  Somesh|2025| 120000|0.4320230387724757|
|  HR|   Anita|2024|  90000|0.9931247493532269|
+----+--------+----+-------+------------------+
only showing top 2 rows


In [112]:
df.withColumn("rand", round(rand() * 100, 1)).show(3)


+-------+--------+----+-------+----+
|   dept|employee|year|revenue|rand|
+-------+--------+----+-------+----+
|     IT|  Somesh|2025| 120000|52.0|
|     HR|   Anita|2024|  90000|70.7|
|Finance|   Karan|2023| 200000|30.8|
+-------+--------+----+-------+----+
only showing top 3 rows


Show duplicates count only

In [ ]:
df.groupBy(df.columns).count().filter(col("count")>1).show()
